# RACE RC Project

### Demo agenda
1. Dataset explanation
2. EDA (Exploratory Data Analysis)
3. Statistical analysis and interpretation
4. Complete preprocessing pipeline
5. Model selection rationale
6. Model training process
7. Model testing/evaluation and metrics interpretation

### Backend implementation in this notebook
- Data pipeline: `src.preprocessing`
- Model A training/evaluation: `src.model_a_train`
- Model B training/evaluation: `src.model_b_train`
- Artifacts and metrics are loaded from `models/*/traditional/metrics_summary.json`


In [ ]:
#clone repo into colab
!git clone https://github.com/Hudiyahaha/Intelligent-Reading-Comprehension-and-Quiz-Generation-System-using-Machine-Learning-Neural-Networks.git

In [4]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
print('Current working directory:', ROOT)
print('Python:', sys.version)


def find_project_root(start: Path) -> Path | None:
    # 1) Walk up from current directory
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / 'requirements.txt').exists() and (p / 'src').exists():
            return p

    # 2) Common Colab clone locations
    candidates = [
        Path('/content/Intelligent-Reading-Comprehension-and-Quiz-Generation-System-using-Machine-Learning-Neural-Networks/race_rc_project'),
        Path('/content/race_rc_project'),
    ]
    for p in candidates:
        if (p / 'requirements.txt').exists() and (p / 'src').exists():
            return p

    # 3) Shallow search under /content for race_rc_project
    base = Path('/content')
    if base.exists():
        for p in base.rglob('race_rc_project'):
            if (p / 'requirements.txt').exists() and (p / 'src').exists():
                return p
    return None

project_root = find_project_root(ROOT)
if project_root is None:
    raise FileNotFoundError(
        'Could not find race_rc_project root. Clone the repo first, then rerun.\n'
        'Example:\n'
        '!git clone https://github.com/Hudiyahaha/Intelligent-Reading-Comprehension-and-Quiz-Generation-System-using-Machine-Learning-Neural-Networks.git\n'
        '%cd /content/Intelligent-Reading-Comprehension-and-Quiz-Generation-System-using-Machine-Learning-Neural-Networks/race_rc_project'
    )

if Path.cwd().resolve() != project_root.resolve():
    %cd {project_root}

print('Project root:', Path.cwd())

Current working directory: D:\23I-0761_23I-0765_AI\race_rc_project
Python: 3.14.0 (tags/v3.14.0:ebf955d, Oct  7 2025, 10:15:03) [MSC v.1944 64 bit (AMD64)]
Project root: D:\23I-0761_23I-0765_AI\race_rc_project


In [ ]:
# Install dependencies 
!pip -q install -r requirements.txt

## 1) Dataset explanation and data access setup

This project expects split-wise CSV files in `data/raw/`:
- `train.csv`: model fitting split
- `dev.csv` or `val.csv`: validation split for tuning/selection
- `test.csv`: held-out split for final evaluation

### Colab data setup options
Copy files from mounting Google Drive to colab

In [ ]:

from pathlib import Path
import shutil

DRIVE_DATA_DIR = '/content/drive/MyDrive/race_data'  # change this path
raw_dir = Path('data/raw')
raw_dir.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    src_dir = Path(DRIVE_DATA_DIR)
    if not src_dir.exists():
        raise FileNotFoundError(f'Drive path not found: {src_dir}')
    for fname in ['train.csv', 'dev.csv', 'val.csv', 'test.csv']:
        src = src_dir / fname
        if src.exists():
            dst = raw_dir / fname
            shutil.copy2(src, dst)
            print('Copied:', src, '->', dst)
except Exception as e:
    print('Drive copy step skipped/failed:', e)
    print('Use Option 1 or fix DRIVE_DATA_DIR and rerun this cell.')

In [ ]:
# Check dataset files
from pathlib import Path

raw_dir = Path('data/raw')
expected = ['train.csv', 'test.csv']
optional_val = ['dev.csv', 'val.csv']

for f in expected:
    print(f, '->', (raw_dir / f).exists())
print('dev/val exists ->', any((raw_dir / f).exists() for f in optional_val))

if not (raw_dir / 'train.csv').exists() or not (raw_dir / 'test.csv').exists() or not any((raw_dir / f).exists() for f in optional_val):
    raise FileNotFoundError('Missing required CSVs in data/raw. Add train/test + dev or val first.')

## 2) EDA and statistical analysis

This section performs quick dataset profiling to support modeling decisions.

It covers:
- Split sizes and schema checks
- Missing values overview
- Text length distribution (proxy for difficulty/complexity)
- Target/answer distribution (class balance)
- Statistical comparison of length distributions across splits

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

raw_dir = Path('data/raw')


def load_split(path):
    if path.exists():
        return pd.read_csv(path)
    return None

train_df = load_split(raw_dir / 'train.csv')
val_df = load_split(raw_dir / 'dev.csv')
if val_df is None:
    val_df = load_split(raw_dir / 'val.csv')
test_df = load_split(raw_dir / 'test.csv')

splits = {
    'train': train_df,
    'val': val_df,
    'test': test_df,
}

for name, df in splits.items():
    if df is None:
        print(f'{name}: missing')
        continue
    print(f'\n{name.upper()}')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print('missing values (top):')
    print(df.isna().sum().sort_values(ascending=False).head(10))

In [ ]:
# Text-length and answer-distribution analysis

def guess_text_col(df):
    if df is None:
        return None
    preferred = ['article', 'context', 'passage', 'question', 'text']
    cols_lower = {c.lower(): c for c in df.columns}
    for p in preferred:
        if p in cols_lower:
            return cols_lower[p]
    obj_cols = [c for c in df.columns if str(df[c].dtype) == 'object']
    return obj_cols[0] if obj_cols else None


def guess_target_col(df):
    if df is None:
        return None
    preferred = ['answer', 'label', 'target', 'correct_option']
    cols_lower = {c.lower(): c for c in df.columns}
    for p in preferred:
        if p in cols_lower:
            return cols_lower[p]
    return None

length_stats = {}
for name, df in splits.items():
    if df is None:
        continue
    tcol = guess_text_col(df)
    acol = guess_target_col(df)

    print(f'\n--- {name.upper()} ---')
    if tcol:
        lengths = df[tcol].fillna('').astype(str).str.split().str.len()
        length_stats[name] = lengths
        print(f'text column used: {tcol}')
        print('word-length stats:', lengths.describe(percentiles=[0.25, 0.5, 0.75]).to_dict())
    else:
        print('No text-like column detected for length analysis.')

    if acol:
        print(f'target column used: {acol}')
        print('target distribution:')
        print(df[acol].value_counts(normalize=True).sort_index())
    else:
        print('No target-like column detected for class-distribution analysis.')

In [ ]:
# Statistical analysis: train vs test text-length difference
try:
    from scipy.stats import ttest_ind, ks_2samp
    have_scipy = True
except Exception:
    have_scipy = False

if 'train' in length_stats and 'test' in length_stats:
    train_len = length_stats['train'].dropna().astype(float)
    test_len = length_stats['test'].dropna().astype(float)

    print('Train mean length:', float(train_len.mean()))
    print('Test mean length :', float(test_len.mean()))

    if have_scipy:
        t_stat, t_p = ttest_ind(train_len, test_len, equal_var=False)
        ks_stat, ks_p = ks_2samp(train_len, test_len)
        print('\nWelch t-test:', {'t_stat': float(t_stat), 'p_value': float(t_p)})
        print('KS test      :', {'ks_stat': float(ks_stat), 'p_value': float(ks_p)})
    else:
        print('\nscipy not available; only descriptive comparison shown.')
else:
    print('Insufficient length stats (need both train and test).')

### Visualizations

Three plots to satisfy the rubric's data-distribution / correlation /
feature-relationship requirements:

1. **Article word-length distribution** per split (overlapping histograms).
2. **Answer-letter frequency** across the four options A/B/C/D.
3. **Correlation heatmap** over the handcrafted lexical features computed by
   `src/preprocessing.py` (read from `data/processed/verify_train.parquet`
   when available).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

ax = axes[0]
colors = {"train": "#1f77b4", "val": "#ff7f0e", "test": "#2ca02c"}
plotted = False
for name, df in splits.items():
    if df is None:
        continue
    col = guess_text_col(df)
    if col is None:
        continue
    lengths = df[col].fillna("").astype(str).str.split().str.len()
    ax.hist(
        lengths.clip(upper=lengths.quantile(0.99)),
        bins=40,
        alpha=0.45,
        label=name,
        color=colors.get(name, None),
    )
    plotted = True
ax.set_title("Article word-length distribution (clipped at 99th pct)")
ax.set_xlabel("words per article")
ax.set_ylabel("count")
if plotted:
    ax.legend()

ax = axes[1]
combined = pd.concat([df for df in splits.values() if df is not None], ignore_index=True)
if "answer" in combined.columns:
    counts = combined["answer"].astype(str).str.upper().value_counts().reindex(list("ABCD")).fillna(0)
    ax.bar(counts.index, counts.values, color=["#4c72b0", "#dd8452", "#55a868", "#c44e52"])
    ax.set_title("Gold answer-letter frequency (all splits)")
    ax.set_xlabel("answer")
    ax.set_ylabel("count")
    total = counts.sum()
    if total > 0:
        for i, v in enumerate(counts.values):
            ax.text(i, v, f"{v/total:.1%}", ha="center", va="bottom", fontsize=9)
else:
    ax.set_title("No 'answer' column to plot")

ax = axes[2]
verify_path = Path("data/processed/verify_train.parquet")
if verify_path.is_file():
    v = pd.read_parquet(verify_path)
    feat_cols = [c for c in v.columns if c.startswith("feat_")]
    if feat_cols:
        corr = v[feat_cols].astype(float).corr()
        im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap="coolwarm")
        ax.set_xticks(range(len(feat_cols)))
        ax.set_yticks(range(len(feat_cols)))
        ax.set_xticklabels([c.replace("feat_", "") for c in feat_cols], rotation=45, ha="right", fontsize=8)
        ax.set_yticklabels([c.replace("feat_", "") for c in feat_cols], fontsize=8)
        ax.set_title("Handcrafted feature correlation (verify_train)")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        for i in range(len(feat_cols)):
            for j in range(len(feat_cols)):
                ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=7, color="black")
    else:
        ax.set_title("No feat_ columns in verify_train.parquet")
else:
    ax.set_title("verify_train.parquet not found — run preprocessing first")

plt.tight_layout()
plt.show()

### Outlier detection (IQR-based)

The Tukey 1.5×IQR rule on `article` word-count flags the long-tail passages
that would otherwise dominate vector-norm-based features. We report the
number of flagged rows, the IQR bounds, and a boxplot of the train split so
the cutoff is auditable.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

outlier_rows = []
length_by_split: dict[str, pd.Series] = {}
for name, df in splits.items():
    if df is None:
        continue
    col = guess_text_col(df)
    if col is None:
        continue
    lengths = df[col].fillna("").astype(str).str.split().str.len().astype(float)
    length_by_split[name] = lengths
    q1, q3 = lengths.quantile(0.25), lengths.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flagged = lengths[(lengths < lo) | (lengths > hi)]
    outlier_rows.append(
        {
            "split": name,
            "n": int(len(lengths)),
            "q1": float(q1),
            "median": float(lengths.median()),
            "q3": float(q3),
            "iqr": float(iqr),
            "lower_bound": float(lo),
            "upper_bound": float(hi),
            "outlier_count": int(len(flagged)),
            "outlier_pct": float(len(flagged) / max(1, len(lengths))),
            "max_value": float(lengths.max()),
            "min_value": float(lengths.min()),
        }
    )

outliers_df = pd.DataFrame(outlier_rows)
print("IQR (1.5x) outlier summary on article word-counts:\n")
print(outliers_df.to_string(index=False, float_format=lambda v: f"{v:.2f}"))

if length_by_split:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.boxplot(
        [length_by_split[k].values for k in length_by_split.keys()],
        tick_labels=list(length_by_split.keys()),
        showfliers=True,
    )
    ax.set_title("Article word-count boxplot per split (outliers shown)")
    ax.set_ylabel("words per article")
    plt.tight_layout()
    plt.show()

## 3) Interpretation of EDA and statistical analysis

- If train/test length means are close and statistical tests are non-significant (`p > 0.05`), split shift risk is lower.
- If tests are significant, mention potential distribution shift and expected impact on generalization.
- Use target distribution to justify balancing or weighting strategies during modeling.
- Missing-value findings directly motivate preprocessing decisions in the next section.

## 4) Complete data preprocessing steps

The preprocessing module (`src.preprocessing`) performs end-to-end preparation before model training.

Checklist:
- Validate required split files (`train`, `dev/val`, `test`)
- Clean and normalize fields
- Build structured processed datasets and split manifests
- Generate feature spaces (e.g., OHE/TF-IDF dimensions in manifest)
- Save versioned artifacts in `data/processed/` for reproducible training

In [ ]:
# Run preprocessing
!python -m src.preprocessing --raw-dir data/raw --processed-dir data/processed --combined-split --train-fraction 0.8 --val-fraction-of-train-pool 0.1
# !python -m src.preprocessing --raw-dir data/raw --processed-dir data/processed

In [ ]:
# Inspect preprocessing outputs
manifest_path = Path('data/processed/manifest.json')
if not manifest_path.exists():
    raise FileNotFoundError('manifest.json not found. Preprocessing may have failed.')

manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
print(json.dumps({
    'mcq_splits': manifest.get('mcq_splits'),
    'verify_splits': manifest.get('splits'),
    'ohe_feature_dim': manifest.get('ohe_feature_dim'),
    'tfidf_feature_dim': manifest.get('tfidf_feature_dim')
}, indent=2))

## 5) Model selection process and training plan

### Why this model family is selected
- This project uses traditional ML pipelines for reproducibility and lower compute cost.
- Model A combines supervised modeling with a clustering-assisted strategy for robust question generation quality.
- Model B is kept as an alternative baseline/competitor under the same preprocessing pipeline.

### Training process
- Train on processed train split
- Monitor on validation split
- Keep test split fully held-out for final reporting

In [ ]:
# Train Model A (full)
!python -m src.model_a_train --processed-dir data/processed --output-dir models/model_a/traditional

In [ ]:
# Faster debug run 
# !python -m src.model_a_train --processed-dir data/processed --output-dir models/model_a/traditional_debug --generation-max-train-mcq 5000 --generation-max-val-mcq 2000 --generation-max-test-mcq 2000 --max-eval-mcq 500

## 6) Model testing/evaluation results (Model A)

After training, this section reads saved evaluation outputs and interprets BLEU/ROUGE/METEOR on validation and test splits.

- Generalization gap (validation vs test)
- Whether quality is stable enough for production use
- Which artifacts are generated for downstream backend integration

In [ ]:
# Read Model A metrics (traditional generation: BLEU / ROUGE / METEOR — no 'results' key)
metrics_path = Path('models/model_a/traditional/metrics_summary.json')
if not metrics_path.exists():
    raise FileNotFoundError('Model A metrics file not found. Train step may have failed.')

metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
print('Model:', metrics.get('model'))
print('Evaluation:', metrics.get('evaluation'))
print('\nConfig:')
print(json.dumps(metrics.get('config', {}), indent=2))
print('\nValidation (BLEU / ROUGE / METEOR):')
print(json.dumps(metrics.get('validation', {}), indent=2))
print('\nTest (BLEU / ROUGE / METEOR):')
print(json.dumps(metrics.get('test', {}), indent=2))
print('\nArtifacts:')
print(json.dumps(metrics.get('artifacts', {}), indent=2))

In [ ]:
# Quick artifact check
artifact_dir = Path('models/model_a/traditional')
for p in sorted(artifact_dir.glob('*')):
    print(p.name)

## 7) Model B training and evaluation

Model B is trained as an alternative backend approach to compare with Model A under the same processed data.


In [ ]:
# Train Model B 
!python -m src.model_b_train --processed-dir data/processed --output-dir models/model_b/traditional

In [ ]:
# Faster Model B debug run
# !python -m src.model_b_train --processed-dir data/processed --output-dir models/model_b/traditional_debug --max-train-mcq 5000 --max-val-mcq 2000 --max-test-mcq 2000

In [ ]:
# Read Model B metrics
metrics_b_path = Path('models/model_b/traditional/metrics_summary.json')
if not metrics_b_path.exists():
    raise FileNotFoundError('Model B metrics file not found. Train step may have failed.')

metrics_b = json.loads(metrics_b_path.read_text(encoding='utf-8'))
print('Model:', metrics_b.get('model'))
print('\nConfig:')
print(json.dumps(metrics_b.get('config', {}), indent=2))
print('\nValidation metrics:')
print(json.dumps(metrics_b.get('validation', {}), indent=2))
print('\nTest metrics:')
print(json.dumps(metrics_b.get('test', {}), indent=2))

In [ ]:
# Quick Model B artifact check
artifact_b_dir = Path('models/model_b/traditional')
for p in sorted(artifact_b_dir.glob('*')):
    print(p.name)

## 8) Comparative performance summary and final interpretation

This final section compares Model A and Model B on saved metrics and provides an interpretation for final backend recommendation.

In [ ]:
from pathlib import Path
import json

model_a_path = Path('models/model_a/traditional/metrics_summary.json')
model_b_path = Path('models/model_b/traditional/metrics_summary.json')

if not model_a_path.exists() or not model_b_path.exists():
    print('Run both Model A and Model B training/evaluation cells first.')
else:
    ma = json.loads(model_a_path.read_text(encoding='utf-8'))
    mb = json.loads(model_b_path.read_text(encoding='utf-8'))

    print('=== Model comparison (test split) ===')
    print('Model A test metrics:')
    print(json.dumps(ma.get('test', {}), indent=2))
    print('\nModel B test metrics:')
    print(json.dumps(mb.get('test', {}), indent=2))

    print('\nInterpretation guide:')
    print('- Higher BLEU/ROUGE/METEOR indicates better overlap with reference questions.')
    print('- Prefer the model with consistently stronger test metrics, not only validation metrics.')
    print('- If metrics are close, choose the simpler/faster model for deployment stability.')